In [4]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient("mongodb", 27017)

# 2. Crear (o usar) la base de datos y la colección
db = client['ArriendosPrueba_IVRegión'] 
coleccion = db['G2_PuebaChilePropiedades2'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [6]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda: ChilePropiedades (Arriendos IV Región)
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# --- 🎯 ASIGNACIÓN DE PÁGINAS ---
PAGINA_INICIO = 1
PAGINA_FIN = 1

os.environ["DISPLAY"] = ":99"
os.system("pkill -9 chrome && pkill -9 chromedriver")

print(f"🧹 Entorno listo. ChilePropiedades: Páginas {PAGINA_INICIO} a {PAGINA_FIN}")

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)

    # ==============================================================================
    # FASE 1: RECOLECCIÓN DE URLs
    # ==============================================================================

    url_base = "https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/coquimbo/"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):

        print(f"\n--- 📄 Página {pagina_actual} ---")

        url_pagina = f"{url_base}?pagina={pagina_actual}"

        driver.get(url_pagina)

        time.sleep(5)

        # Scroll para cargar contenido dinámico
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

        print("Página cargada:", driver.title)

        links = driver.find_elements(By.TAG_NAME, "a")

        print("Links encontrados:", len(links))

        for link in links:
            try:
                url = link.get_attribute("href")

                if not url:
                    continue

                # SOLO URLs de propiedades reales
                if "/ver-publicacion/" not in url:
                    continue

                catalogo_urls.append({
                    "url": url,
                    "ubicacion": "Coquimbo"
                })

            except:
                continue


    # Eliminar duplicados
    catalogo_urls = list({v['url']: v for v in catalogo_urls}.values())

    print(f"\n🏠 Catálogo listo: {len(catalogo_urls)} propiedades")


    # ==============================================================================
    # FASE 2: INSPECCIÓN DE PROPIEDADES
    # ==============================================================================

    for i, propiedad in enumerate(catalogo_urls):

        print(f"\nProcesando {i+1}/{len(catalogo_urls)}")
        print("URL:", propiedad["url"])

        registro = {
            "responsable": "Constanza Torres",
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": "",
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": 0.0,
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"]
        }

        try:

            driver.get(propiedad["url"])
            time.sleep(4)

            texto_total = driver.find_element(By.TAG_NAME, "body").text.lower()

            # ========================
            # TITULO
            # ========================

            try:
                titulo = driver.find_element(By.TAG_NAME, "h1").text
                registro["titulo"] = titulo
            except:
                registro["titulo"] = "Propiedad ChilePropiedades"


            # ========================
            # PRECIO
            # ========================

            match_precio = re.search(r'\$\s?([\d\.]+)', texto_total)

            if match_precio:
                precio = match_precio.group(1).replace(".", "")
                registro["precio"] = float(precio)


            # ========================
            # METROS CUADRADOS
            # ========================

            match_m2 = re.search(r'(\d+)\s*(m2|m²)', texto_total)

            if match_m2:
                registro["m2"] = int(match_m2.group(1))


            # ========================
            # DORMITORIOS
            # ========================

            match_dorm = re.search(r'(\d+)\s*(dormitorio|habitacion)', texto_total)

            if match_dorm:
                registro["dormitorios"] = int(match_dorm.group(1))


            # ========================
            # BAÑOS
            # ========================

            match_bano = re.search(r'(\d+)\s*baño', texto_total)

            if match_bano:
                registro["baños"] = int(match_bano.group(1))


            # ========================
            # ESTACIONAMIENTO
            # ========================

            match_est = re.search(r'(\d+)\s*estacionamiento', texto_total)

            if match_est:
                registro["estacionamiento"] = int(match_est.group(1))
            elif "estacionamiento" in texto_total:
                registro["estacionamiento"] = 1


            # ========================
            # AMENIDADES
            # ========================

            if "piscina" in texto_total:
                registro["piscina"] = 1

            if "quincho" in texto_total:
                registro["quincho"] = 1

            if "terraza" in texto_total:
                registro["terraza"] = 1

            if "gimnasio" in texto_total:
                registro["gimnasio"] = 1

            if "lavander" in texto_total:
                registro["lavanderia"] = 1


            # ========================
            # CREAR TITULO INTELIGENTE
            # ========================

            if registro["titulo"] == "" or "departamentos" in registro["titulo"].lower():

                titulo_auto = "Propiedad ChilePropiedades"

                if registro["dormitorios"] > 0:
                    titulo_auto += f" - {registro['dormitorios']}D"

                if registro["baños"] > 0:
                    titulo_auto += f" {registro['baños']}B"

                titulo_auto += f" - {registro['ubicacion']}"

                registro["titulo"] = titulo_auto


            # ========================
            # FILTROS
            # ========================

            # eliminar arriendo diario
            if "por noche" in texto_total or "por día" in texto_total:
                print("Descartado arriendo diario")
                continue


            # eliminar precios muy bajos
            if registro["precio"] != 0 and registro["precio"] < 200000:
                print("Descartado precio muy bajo")
                continue


            datos_finales.append(registro)


        except Exception as e:
            print("Error en propiedad:", e)

        time.sleep(2)


except Exception as e:
    print("Error general:", e)


finally:
    driver.quit()


# ==============================================================================
# GUARDADO MONGO
# ==============================================================================

print("\n💾 Guardando Mongo")

client = MongoClient("mongodb",27017)

db = client["ArriendosPrueba_IVRegión"]

coleccion = db["G2_PruebaChilePropiedades2"]

for d in datos_finales:
    coleccion.update_one(
        {"url": d["url"]},
        {"$set": d},
        upsert=True
    )

print("\n✅ Extracción terminada:", len(datos_finales))

🧹 Entorno listo. ChilePropiedades: Páginas 1 a 1

--- 📄 Página 1 ---
Página cargada: Departamentos en Arriendo en Coquimbo
Links encontrados: 95

🏠 Catálogo listo: 10 propiedades

Procesando 1/10
URL: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/coquimbo/departamento/av-los-pescadores-5341/106187612

Procesando 2/10
URL: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/coquimbo/departamento/juan-antonio-rios-404/116184695

Procesando 3/10
URL: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/coquimbo/departamento/condomio-costa-del-elqui/71862939

Procesando 4/10
URL: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/coquimbo/departamento/condominio-paseo-san-carlos-ii/115997465

Procesando 5/10
URL: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/coquimbo/departamento/condominio-costa-de-elqui/115997419

Procesando 6/10
URL: https://chilepropiedades.cl/ver-publicacion/arriendo-mensual/coquimbo/departamento/penthouse-vista